# 🧠 Sentiment Analysis with NLP
### A Comprehensive End-to-End Guide: Theory + Code

---

> **What you will learn in this notebook:**
> 1. What Sentiment Analysis is and why it matters
> 2. Understanding the Sentiment140 dataset
> 3. Text preprocessing pipeline — step by step
> 4. Feature engineering with TF-IDF
> 5. Training and comparing three classifiers
> 6. Evaluating and interpreting results
> 7. Building a real-time prediction function

---

## 📖 Section 1 — What is Sentiment Analysis?

**Sentiment Analysis** (also called *Opinion Mining*) is a Natural Language Processing (NLP) technique used to automatically identify and extract subjective information from text — most commonly the **emotional tone** or **polarity** of the writer.

### 🎯 Why Does It Matter?
Companies, researchers, and governments use sentiment analysis to:
- Monitor **brand reputation** on social media
- Understand **customer feedback** from reviews
- Detect **public opinion** during elections or crises
- Prioritize **customer support tickets** by urgency

### 📊 Types of Sentiment

| Type | Description | Example |
|------|-------------|--------|
| **Positive** | Expresses praise, satisfaction, or joy | *"I absolutely loved this product!"* |
| **Neutral** | Factual or emotionally flat | *"The product arrived on Tuesday."* |
| **Negative** | Expresses frustration, anger, or disappointment | *"Terrible experience, never again."* |

### 🛠️ How Does It Work (High Level)?
```
Raw Text → Cleaning → Tokenization → Feature Extraction → ML Model → Sentiment Label
```

In this notebook we follow exactly this pipeline using real Twitter data.

## 📦 Section 2 — Dataset Description

We use the **Sentiment140** dataset — a classic benchmark for tweet-level sentiment analysis.

### 📋 Dataset Structure

| Column | Description | Example |
|--------|-------------|--------|
| `sentiment` | Polarity label: **0** = Negative, **2** = Neutral, **4** = Positive | `0` |
| `id` | Unique tweet identifier | `2087` |
| `date` | Timestamp of the tweet | `Sat May 16 23:58:44 UTC 2009` |
| `query` | Search query used to retrieve the tweet (`NO_QUERY` if none) | `lyx` |
| `user` | Twitter username | `robotickilldozr` |
| `text` | The actual tweet content | `Lyx is cool` |

### 🔑 Key Facts
- **1,600,000 tweets** in the full version
- Labels were assigned **automatically** using emoticons (`:)` → Positive, `:(` → Negative)
- No manual annotation — making it a **weakly supervised** dataset
- Emoticons are **removed** from the text (to prevent leakage)

### ⚠️ Loading Note
This notebook is structured for the Kaggle environment. Replace the paths below with your local paths if running elsewhere:
```
kagglehub.dataset_download('abhi8923shriv/sentiment-analysis-dataset')
```

## ⚙️ Section 3 — Import Libraries

We import all necessary libraries upfront. Here's what each one does:

| Library | Purpose |
|---------|--------|
| `numpy`, `pandas` | Data manipulation |
| `matplotlib`, `seaborn` | Visualizations |
| `re` | Regular expressions for text cleaning |
| `nltk` | NLP toolkit — tokenization, stopwords, stemming, lemmatization |
| `sklearn` | Machine learning models, vectorization, and evaluation |
| `string` | Python's built-in punctuation list |

In [33]:
# ── Standard Libraries ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
import os

warnings.filterwarnings('ignore')

# ── NLTK: Natural Language Toolkit ──────────────────────────────────────────
import nltk
nltk.download('punkt',        quiet=True)  # Sentence/word tokenizer
nltk.download('stopwords',   quiet=True)  # Common English stop words
nltk.download('wordnet',     quiet=True)  # WordNet lexical database
nltk.download('punkt_tab',   quiet=True)  # Updated punkt tokenizer tables

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import LancasterStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.probability import FreqDist

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 📂 Section 4 — Load the Dataset

We load both train and test CSVs, then concatenate them into a single DataFrame for unified preprocessing.

**Why `encoding='latin1'`?**  
The Sentiment140 CSV was created in 2009 and contains characters outside standard ASCII (e.g., accented letters, special symbols). `latin1` (ISO-8859-1) handles these gracefully without throwing `UnicodeDecodeError`.

In [34]:
# ── Load Train and Test CSVs ─────────────────────────────────────────────────
# Update these paths if you're running locally

# --- FIX: Download dataset for Colab if not in Kaggle environment ---
# Install Kaggle API client
!pip install -q kaggle

# Create .kaggle directory and copy API key (requires user to upload kaggle.json)
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset (replace 'abhi8923shriv/sentiment-analysis-dataset' with actual dataset slug if different)
!kaggle datasets download -d abhi8923shriv/sentiment-analysis-dataset

# Unzip the dataset files
!unzip -o sentiment-analysis-dataset.zip

# Now set the paths to the unzipped files in the current directory
TRAIN_PATH = 'train.csv'
TEST_PATH  = 'test.csv'
# --- End of FIX ---

train_data = pd.read_csv(TRAIN_PATH, encoding='latin1')
test_data  = pd.read_csv(TEST_PATH,  encoding='latin1')

print(f"Train shape : {train_data.shape}")
print(f"Test shape  : {test_data.shape}")

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/abhi8923shriv/sentiment-analysis-dataset
License(s): CC0-1.0
sentiment-analysis-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  sentiment-analysis-dataset.zip
  inflating: test.csv                
  inflating: testdata.manual.2009.06.14.csv  
  inflating: train.csv               
  inflating: training.1600000.processed.noemoticon.csv  
Train shape : (27481, 10)
Test shape  : (4815, 9)


In [35]:
# ── Combine into one DataFrame ────────────────────────────────────────────────
# We concatenate so all preprocessing is applied uniformly to every sample.
df = pd.concat([train_data, test_data], ignore_index=True)

# Configure pandas to show all columns and avoid truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

print(f"Combined shape: {df.shape}")
df.head()

Combined shape: (32296, 10)


,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346.0,652860.0,60.0
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797.0,27400.0,105.0
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044.0,2381740.0,18.0
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265.0,470.0,164.0
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272.0,1246700.0,26.0


In [36]:
# ── Inspect Data Types and Missing Values ─────────────────────────────────────
# This tells us which columns contain text vs. numerics, and where nulls exist.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32296 entries, 0 to 32295
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   textID            31015 non-null  object 
 1   text              31014 non-null  object 
 2   selected_text     27480 non-null  object 
 3   sentiment         31015 non-null  object 
 4   Time of Tweet     31015 non-null  object 
 5   Age of User       31015 non-null  object 
 6   Country           31015 non-null  object 
 7   Population -2020  31015 non-null  float64
 8   Land Area (Km²)   31015 non-null  float64
 9   Density (P/Km²)   31015 non-null  float64
dtypes: float64(3), object(7)
memory usage: 2.5+ MB


In [37]:
# ── Check for Missing Values ─────────────────────────────────────────────────
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
textID              1281
text                1282
selected_text       4816
sentiment           1281
Time of Tweet       1281
Age of User         1281
Country             1281
Population -2020    1281
Land Area (Km²)     1281
Density (P/Km²)     1281
dtype: int64


## 🧹 Section 5 — Text Preprocessing Pipeline

Raw tweet text is messy. Before any model can learn from it, we must clean and normalize it through several stages. Think of preprocessing as **signal extraction** — we remove noise so the model can focus on meaning.

### The Full Pipeline
```
Raw Text
  ↓
[1] Remove URLs, HTML tags, special characters
  ↓
[2] Normalize: lowercase, strip extra whitespace
  ↓
[3] Tokenize: split into individual words (tokens)
  ↓
[4] Remove Stopwords: drop common words with no sentiment value
  ↓
[5] Stemming / Lemmatization: reduce words to root form
  ↓
Clean Text ✅
```

### 5.1 — Remove Unnecessary Characters

**Why?** Tweets contain URLs (`http://...`), HTML tags (`<br>`), special characters (`@`, `#`, `&`), and inconsistent whitespace. These are **syntactic noise** — they carry no sentiment signal for our models.

We use **Regular Expressions (regex)** for pattern-based replacement:
- `<.*?>` — matches any HTML tag (non-greedy)
- `[^a-zA-Z0-9\s]` — matches anything that is NOT a letter, digit, or space
- `\s+` — matches one or more whitespace characters

In [38]:
def remove_unnecessary_characters(text: str) -> str:
    """
    Cleans raw tweet text by removing:
      - HTML tags   (e.g. <br>, <p>)
      - Non-alphanumeric chars  (e.g. @, #, !, emoticons)
      - Extra whitespace

    Parameters
    ----------
    text : str  — The raw tweet string.

    Returns
    -------
    str  — Cleaned text containing only letters, digits, and single spaces.
    """
    text = re.sub(r'<.*?>',          '',  str(text))  # Strip HTML tags
    text = re.sub(r'[^a-zA-Z0-9\s]', '', str(text))  # Keep only alphanumeric
    text = re.sub(r'\s+',            ' ', str(text)).strip()  # Collapse whitespace
    return text

# Apply to the 'text' column
df['clean_text'] = df['text'].apply(remove_unnecessary_characters)

# Preview the effect
print("Original  :", df['text'].iloc[0])
print("Cleaned   :", df['clean_text'].iloc[0])

Original  :  I`d have responded, if I were going
Cleaned   : Id have responded if I were going


### 5.2 — Text Normalization

**Why?** Without normalization, `"Happy"`, `"happy"`, and `"HAPPY"` are treated as **three different words** by the model. Lowercasing makes the vocabulary consistent and smaller.

Normalization steps:
1. **Lowercase** all characters
2. **Strip punctuation** (anything that's not a word character or space)
3. **Remove extra whitespace**

In [39]:
def normalize_text(text) -> str:
    """
    Normalizes text to lowercase and removes punctuation and extra spaces.
    Handles non-string inputs gracefully by converting them first.
    """
    if isinstance(text, str):
        text = text.lower()                          # Lowercase
        text = re.sub(r'[^\w\s]', '', text)          # Remove punctuation
        text = re.sub(r'\s+', ' ', text).strip()     # Collapse whitespace
    else:
        text = str(text)  # Coerce non-strings (e.g. floats from NaN)
    return text

df['normalized_text'] = df['text'].apply(normalize_text)

print("Original   :", df['text'].iloc[1])
print("Normalized :", df['normalized_text'].iloc[1])

Original   :  Sooo SAD I will miss you here in San Diego!!!
Normalized : sooo sad i will miss you here in san diego


### 5.3 — Tokenization

**Why?** Most NLP algorithms work on **individual words** (or subwords), not full strings. Tokenization is the process of splitting a sentence into its constituent **tokens** (words, punctuation marks, etc.).

**NLTK's `word_tokenize`** uses a Punkt tokenizer — it handles edge cases like contractions (`"don't"` → `["do", "n't"]`) and abbreviations better than a simple `str.split()`.

**Example:**
```
Input : "I love NLP!"
Output: ['I', 'love', 'NLP', '!']
```

In [ ]:
def tokenize_text(text) -> list:
    """
    Splits text into a list of word tokens using NLTK's Punkt tokenizer.
    Returns an empty list if tokenization fails (e.g., empty string).
    """
    try:
        return word_tokenize(str(text))
    except Exception as e:
        print(f"Tokenization error: {e}")
        return []

df['tokens'] = df['text'].apply(tokenize_text)

# Show a sample
print("Original :", df['text'].iloc[2])
print("Tokens   :", df['tokens'].iloc[2])

### 5.4 — Remove Stopwords

**Why?** Stopwords are extremely common words — `"the"`, `"is"`, `"at"`, `"which"` — that appear in almost every sentence regardless of sentiment. They add **vocabulary noise** without contributing to meaning.

**NLTK's stopword list** contains 179 English stopwords. After removing them, only the **content words** (nouns, verbs, adjectives) remain, which carry the real sentiment signal.

**Example:**
```
Input : "I am not feeling happy at all today"
Output: "feeling happy today"      ← much more signal-dense!
```

In [ ]:
# Load English stopwords from NLTK
STOP_WORDS = set(stopwords.words('english'))

def remove_stopwords(text) -> str:
    """
    Removes NLTK English stopwords from a text string.
    Non-string inputs are returned as empty strings.
    """
    if isinstance(text, str):
        words = text.split()
        filtered = [w for w in words if w.lower() not in STOP_WORDS]
        return ' '.join(filtered)
    return ''

df['text_without_stopwords'] = df['text'].apply(remove_stopwords)

print("Original        :", df['text'].iloc[3])
print("No Stopwords    :", df['text_without_stopwords'].iloc[3])

### 5.5 — Stemming with Lancaster Stemmer

**Why?** Words like `"running"`, `"runs"`, and `"ran"` all refer to the same concept. **Stemming** reduces words to their root (stem) form so the model treats them as one.

**Lancaster Stemmer** is one of the most aggressive English stemmers — it strips suffixes aggressively:
- `running` → `run`
- `happiness` → `happy`  *(sometimes overstemms)*
- `fishing` → `fish`

> ⚠️ Stemmers can produce non-dictionary words (e.g., `"stripes"` → `"strip"`). For production systems, **Lemmatization** (Section 5.6) is preferred.

In [ ]:
stemmer   = LancasterStemmer()
lemmatizer = WordNetLemmatizer()

def stem_text(text: str) -> str:
    """
    Applies Lancaster stemming to each token in the text.
    Example: 'running quickly' → 'run quick'
    """
    tokens = word_tokenize(str(text))
    stemmed = [stemmer.stem(t) for t in tokens]
    return ' '.join(stemmed)

def lemmatize_text(text: str) -> str:
    """
    Applies WordNet lemmatization to each token.
    More linguistically accurate than stemming — always produces real words.
    Example: 'running' → 'running' (noun form); with pos='v' → 'run'
    """
    tokens = word_tokenize(str(text))
    lemmatized = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(lemmatized)

# Demonstrate the difference
sample = "The dogs were running and barking happily"
print(f"Original    : {sample}")
print(f"Stemmed     : {stem_text(sample)}")
print(f"Lemmatized  : {lemmatize_text(sample)}")

### 5.6 — Drop Missing Values

After all the transformations, some rows might have become empty strings or NaN values (e.g., tweets that were entirely emoji or URLs). We drop them to prevent errors during model training.

In [ ]:
before = len(df)
df.dropna(inplace=True)
after = len(df)

print(f"Rows before drop : {before:,}")
print(f"Rows after drop  : {after:,}")
print(f"Rows removed     : {before - after:,}")

## 📊 Section 6 — Exploratory Data Analysis (EDA)

Before building models, we visualize the data to understand its structure, class balance, and vocabulary characteristics. Good EDA prevents surprises during training.

### 6.1 — Sentiment Distribution

A **class imbalance** (e.g., 90% positive, 10% negative) can cause a model to cheat by always predicting the majority class. We check if our three classes are balanced.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Bar Chart ────────────────────────────────────────────────────────────────
counts = df['sentiment'].value_counts()
axes[0].bar(
    counts.index.astype(str), counts.values,
    color=['#e74c3c', '#3498db', '#2ecc71'], edgecolor='white'
)
axes[0].set_title('Sentiment Class Distribution (Counts)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment (0=Neg, 2=Neu, 4=Pos)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

# ── KDE / Histogram ──────────────────────────────────────────────────────────
# Encode sentiment to numbers for seaborn
sentiment_num = df['sentiment'].map({0: 0, 2: 1, 4: 2})
sns.histplot(sentiment_num, kde=True, color='steelblue', bins=5, ax=axes[1])
axes[1].set_title('Sentiment Distribution (Histogram + KDE)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentiment (encoded)')
axes[1].set_xticks([0, 1, 2])
axes[1].set_xticklabels(['Negative (0)', 'Neutral (2)', 'Positive (4)'])

plt.tight_layout()
plt.show()

# Normalized percentages
print("\nClass proportions:")
print(df['sentiment'].value_counts(normalize=True).rename({0:'Negative', 2:'Neutral', 4:'Positive'}).to_string())

### 6.2 — Sentiment Encoding (Category Codes)

Many sklearn algorithms require **integer labels**, not string labels. We encode the `sentiment` column to numeric category codes:
- 0 (Negative) → code `0`
- 2 (Neutral)  → code `1`
- 4 (Positive) → code `2`

In [ ]:
df['sentiment_code'] = df['sentiment'].astype('category').cat.codes

print("Sentiment → Code mapping:")
print(df[['sentiment', 'sentiment_code']].drop_duplicates().sort_values('sentiment'))

# Quick bar chart of the encoded distribution
df['sentiment_code'].value_counts().sort_index().plot(
    kind='bar', color=['#e74c3c', '#3498db', '#2ecc71'],
    title='Encoded Sentiment Distribution', rot=0
)
plt.xlabel('Sentiment Code (0=Neg, 1=Neu, 2=Pos)')
plt.tight_layout()
plt.show()

### 6.3 — Word Frequency Distribution

**FreqDist** (Frequency Distribution) counts how often each word appears across the entire corpus. Plotting the top-N most frequent words reveals dominant vocabulary and can surface unexpected patterns (e.g., a particular word dominating).  

This is a diagnostic step — not used directly in modeling — but invaluable for understanding your dataset.

In [ ]:
# Tokenize the cleaned text and build a frequency distribution
all_tokens = word_tokenize(' '.join(df['text_without_stopwords'].dropna().astype(str)))
word_freq  = FreqDist(all_tokens)

plt.figure(figsize=(12, 5))
word_freq.plot(30, cumulative=False, title='Top 30 Word Frequency Distribution')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

print("\nTop 10 most frequent words:")
for word, freq in word_freq.most_common(10):
    print(f"  '{word}' : {freq:,}")

## 🔧 Section 7 — Feature Engineering: TF-IDF Vectorization

Machine learning models work with **numbers**, not text. We must convert our cleaned text into a **numerical representation**. We use **TF-IDF** — one of the most effective approaches for text classification.

### 📐 What is TF-IDF?

**TF-IDF** stands for **Term Frequency – Inverse Document Frequency**.

| Component | Formula | Meaning |
|-----------|---------|--------|
| **TF** (Term Frequency) | `count(word in doc) / total_words_in_doc` | How often a word appears in *this* tweet |
| **IDF** (Inverse Document Frequency) | `log(total_docs / docs_containing_word)` | How rare the word is *across all tweets* |
| **TF-IDF** | `TF × IDF` | High score = word is frequent HERE but rare elsewhere → informative! |

### 💡 Intuition
- A word like `"the"` has high TF (common in every tweet) but very low IDF (common everywhere) → **low TF-IDF score** → filtered out
- A word like `"devastated"` appears rarely across tweets but when it does, it's meaningful → **high TF-IDF score** → strong sentiment signal

### 🧮 The Result
A **sparse matrix** where:
- Each **row** = one tweet
- Each **column** = one unique word in the vocabulary
- Each **cell** = the TF-IDF weight of that word in that tweet

In [ ]:
# ── Advanced text preprocessing for the ML pipeline ──────────────────────────
def preprocess_for_model(text: str) -> str:
    """
    Full preprocessing pipeline used before TF-IDF vectorization:
      1. Remove URLs
      2. Remove HTML tags
      3. Remove punctuation
      4. Remove newlines
      5. Remove alphanumeric tokens containing digits
    """
    text = re.sub(r'https?://\S+|www\.\S+', '', text)          # URLs
    text = re.sub(r'<.*?>+',                 '', text)          # HTML tags
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)  # Punctuation
    text = re.sub(r'\n',                     '', text)          # Newlines
    text = re.sub(r'\w*\d\w*',              '', text)          # Words with digits
    return text

# Apply to selected_text (the relevant excerpt) if it exists, else fall back to text
text_col = 'selected_text' if 'selected_text' in df.columns else 'text'
df[text_col] = df[text_col].astype(str).apply(preprocess_for_model)

# ── Define Features (X) and Target (y) ───────────────────────────────────────
X = df[text_col]          # Input: cleaned tweet text
y = df['sentiment']        # Target: sentiment label

print(f"Feature column : '{text_col}'")
print(f"X shape        : {X.shape}")
print(f"y distribution :\n{y.value_counts()}")

In [ ]:
# ── Train / Test Split ────────────────────────────────────────────────────────
# 80% training, 20% testing. random_state=42 ensures reproducibility.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

In [ ]:
# ── TF-IDF Vectorization ──────────────────────────────────────────────────────
# IMPORTANT: We .fit_transform() on TRAIN data only, then .transform() on TEST.
# This prevents data leakage — the model must not see test vocabulary during training.

vectorizer = TfidfVectorizer(
    max_features=50_000,  # Keep only top 50k words by frequency
    ngram_range=(1, 2),   # Include both unigrams and bigrams
    sublinear_tf=True     # Apply log normalization to term frequency
)

XV_train = vectorizer.fit_transform(X_train)   # Fit + transform training set
XV_test  = vectorizer.transform(X_test)        # Transform test set using SAME vocab

print(f"Vocabulary size  : {len(vectorizer.vocabulary_):,} terms")
print(f"XV_train shape   : {XV_train.shape}")
print(f"XV_test shape    : {XV_test.shape}")
print(f"Matrix sparsity  : {1 - XV_train.nnz / (XV_train.shape[0] * XV_train.shape[1]):.4%}")

## 🤖 Section 8 — Machine Learning Models

We train and compare **three classifiers**, each with different inductive biases. Understanding *why* each works differently is as important as knowing the accuracy numbers.

---

### Baseline Score

Before training any model, we establish a **naïve baseline** — the accuracy we'd get by always predicting the most common class. Any model worth using must beat this.

In [ ]:
# Baseline = accuracy of always predicting the majority class
baseline = y.value_counts(normalize=True).max()
print(f"🎯 Naïve Baseline Accuracy: {baseline:.4%}")
print("   (Accuracy of a model that always guesses the most common class)")

### 8.1 — Logistic Regression

**What is it?**  
Despite its name, Logistic Regression is a **classification** algorithm. It learns a linear decision boundary in feature space — a hyperplane that best separates the classes.

**Why use it for NLP?**
- TF-IDF features are **sparse and high-dimensional** — LR handles this naturally
- It is **interpretable**: you can inspect the learned coefficients to see which words drive positive/negative predictions
- It is **fast** and rarely overfits on text data
- `n_jobs=-1` uses all available CPU cores for parallel computation

**How it works:**
```
P(positive | tweet) = sigmoid(w₁·tfidf₁ + w₂·tfidf₂ + ... + wₙ·tfidfₙ + b)
```
The model learns the weights `w` that maximise the likelihood of the correct labels.

In [ ]:
# ── Train Logistic Regression ─────────────────────────────────────────────────
print("Training Logistic Regression...")
lr = LogisticRegression(
    max_iter=500,
    n_jobs=-1,         # Parallelise across all CPU cores
    solver='lbfgs',    # L-BFGS: well-suited for dense, medium-dimensional problems
    C=1.0              # Regularisation strength (1/lambda)
)
lr.fit(XV_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────
pred_lr  = lr.predict(XV_test)
score_lr = accuracy_score(y_test, pred_lr)

print(f"\n✅ Logistic Regression Accuracy: {score_lr:.4%}")
print(f"   Improvement over baseline   : +{(score_lr - baseline):.4%}")
print("\nDetailed Classification Report:")
print(classification_report(y_test, pred_lr, target_names=['Negative', 'Neutral', 'Positive']))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
# Rows = actual labels, Columns = predicted labels
# Diagonal = correct predictions; off-diagonal = mistakes
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_lr,
    display_labels=['Negative', 'Neutral', 'Positive'],
    cmap='Blues', ax=ax
)
ax.set_title('Logistic Regression — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 8.2 — Decision Tree Classifier

**What is it?**  
A Decision Tree learns a tree of **if-then-else rules** that splits the feature space into regions, each assigned to a class.

**How it works:**  
At each node, the tree finds the single feature (e.g., TF-IDF score of the word `"hate"`) that best separates the remaining samples. It splits on that feature until:
- All samples at a leaf are the same class (**pure leaf**), or
- Maximum depth is reached

**Pros:**  
- Fully interpretable (you can visualize the tree)
- No feature scaling required

**Cons:**  
- Tends to **overfit** on high-dimensional sparse data like TF-IDF
- Generally lower accuracy than ensemble methods on text tasks

In [ ]:
print("Training Decision Tree...")
dt = DecisionTreeClassifier(
    max_depth=50,          # Limit depth to reduce overfitting
    min_samples_split=10,  # Require ≥10 samples to split a node
    random_state=42
)
dt.fit(XV_train, y_train)

pred_dt  = dt.predict(XV_test)
score_dt = accuracy_score(y_test, pred_dt)

print(f"\n✅ Decision Tree Accuracy: {score_dt:.4%}")
print(f"   Improvement over baseline: +{(score_dt - baseline):.4%}")
print("\nDetailed Classification Report:")
print(classification_report(y_test, pred_dt, target_names=['Negative', 'Neutral', 'Positive']))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_dt,
    display_labels=['Negative', 'Neutral', 'Positive'],
    cmap='Oranges', ax=ax
)
ax.set_title('Decision Tree — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 8.3 — Random Forest Classifier

**What is it?**  
Random Forest is an **ensemble method** — it builds many decision trees and aggregates their predictions by majority vote.

**Key innovations over a single Decision Tree:**
1. **Bagging (Bootstrap Aggregating):** Each tree is trained on a random subsample of the data → **reduces variance**
2. **Feature Randomness:** At each split, only a random subset of features is considered → **decorrelates trees** so ensemble errors cancel out

**Why it's usually better:**  
Where a single tree overfits, the forest's votes average out noise. It's one of the most robust off-the-shelf classifiers for tabular and text data.

> ⏱️ Training a Random Forest on TF-IDF is slower than LR. Be patient on large datasets!

In [ ]:
print("Training Random Forest... (this may take a moment)")
rfc = RandomForestClassifier(
    n_estimators=100,     # 100 individual decision trees
    max_depth=50,         # Limit depth per tree
    n_jobs=-1,            # Use all CPU cores
    random_state=0
)
rfc.fit(XV_train, y_train)

pred_rfc  = rfc.predict(XV_test)
score_rfc = accuracy_score(y_test, pred_rfc)

print(f"\n✅ Random Forest Accuracy: {score_rfc:.4%}")
print(f"   Improvement over baseline: +{(score_rfc - baseline):.4%}")
print("\nDetailed Classification Report:")
print(classification_report(y_test, pred_rfc, target_names=['Negative', 'Neutral', 'Positive']))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_rfc,
    display_labels=['Negative', 'Neutral', 'Positive'],
    cmap='Greens', ax=ax
)
ax.set_title('Random Forest — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 📈 Section 9 — Model Comparison

Now we compare all models side by side. A good comparison looks at:
- **Accuracy** — overall correctness
- **Precision** — of all predictions of class X, how many were correct?
- **Recall** — of all actual class X instances, how many did we catch?
- **F1-Score** — harmonic mean of precision and recall (useful for imbalanced classes)

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model': ['Baseline (Majority Class)', 'Logistic Regression',
              'Decision Tree', 'Random Forest'],
    'Accuracy': [baseline, score_lr, score_dt, score_rfc]
})
results['Accuracy (%)'] = (results['Accuracy'] * 100).round(2)
results['vs. Baseline'] = ((results['Accuracy'] - baseline) * 100).round(2)
results = results.sort_values('Accuracy', ascending=False).reset_index(drop=True)

print(results[['Model', 'Accuracy (%)', 'vs. Baseline']].to_string(index=False))

In [ ]:
# ── Bar Chart Comparison ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

models    = ['Baseline', 'Logistic\nRegression', 'Decision\nTree', 'Random\nForest']
scores    = [baseline, score_lr, score_dt, score_rfc]
colors    = ['#95a5a6', '#3498db', '#e67e22', '#2ecc71']

bars = ax.bar(models, [s * 100 for s in scores], color=colors,
              edgecolor='white', linewidth=1.5, width=0.5)

# Annotate bars
for bar, score in zip(bars, scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{score:.2%}',
        ha='center', va='bottom', fontweight='bold', fontsize=11
    )

ax.set_ylim(0, 100)
ax.axhline(baseline * 100, color='red', linestyle='--', linewidth=1.5, label='Baseline')
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Accuracy (%)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 🔮 Section 10 — Real-Time Sentiment Prediction

Now that we have trained models, let's build a **prediction function** that can take any piece of text and return its predicted sentiment — this is how a production sentiment analysis service would work.

### How It Works
1. Accept raw text as input
2. Apply the **same preprocessing** used during training (critical — the model expects the same format)
3. Vectorize using the **same TF-IDF vectorizer** (fit on training data)
4. Pass to the chosen model and decode the numeric prediction back to a readable label

In [ ]:
# ── Label decoder ─────────────────────────────────────────────────────────────
LABEL_MAP = {
    0: "😠 NEGATIVE",
    2: "😐 NEUTRAL",
    4: "😊 POSITIVE"
}

def predict_sentiment(text: str, model=lr) -> str:
    """
    Predicts the sentiment of an input text using a trained model.

    Pipeline:
      1. Preprocess  — clean URLs, HTML, punctuation, digits
      2. Vectorize   — apply TF-IDF using the training-fitted vectorizer
      3. Predict     — obtain the numeric class from the model
      4. Decode      — map the numeric class back to a readable label

    Parameters
    ----------
    text  : str   — Raw input text to classify.
    model : sklearn classifier — Default is Logistic Regression.

    Returns
    -------
    str  — Human-readable sentiment label.
    """
    # Step 1: Preprocess
    cleaned = preprocess_for_model(text)

    # Step 2: Vectorize (must use the same vectorizer — never refit!)
    vector = vectorizer.transform([cleaned])

    # Step 3: Predict
    prediction = model.predict(vector)[0]

    # Step 4: Decode
    return LABEL_MAP.get(prediction, f"Unknown label: {prediction}")

print("✅ Prediction function ready!")

In [ ]:
# ── Run Predictions on Sample Texts ──────────────────────────────────────────
test_sentences = [
    "I absolutely love this! Best day ever!",
    "This is terrible. I'm so disappointed and frustrated.",
    "The package arrived on Monday.",
    "Not sure how I feel about this honestly.",
    "Amazing product, highly recommend to everyone!",
    "Worst experience of my life. Never again.",
    "I went to the store and bought some milk.",
]

print(f"{'Text':<55} {'Sentiment':>15}")
print("-" * 72)
for sentence in test_sentences:
    label = predict_sentiment(sentence, model=lr)
    print(f"{sentence[:53]:<55} {label:>15}")

In [ ]:
# ── Compare predictions across all three models ───────────────────────────────
print(f"\n{'Text':<45} {'Log. Reg':>12} {'Dec. Tree':>12} {'Rnd. Forest':>12}")
print("-" * 85)
for sentence in test_sentences[:5]:
    lr_pred  = predict_sentiment(sentence, model=lr)
    dt_pred  = predict_sentiment(sentence, model=dt)
    rfc_pred = predict_sentiment(sentence, model=rfc)
    print(f"{sentence[:43]:<45} {lr_pred:>12} {dt_pred:>12} {rfc_pred:>12}")

## 🏁 Section 11 — Conclusion & Key Takeaways

### 📊 Final Results Summary

| Model | Accuracy | Observation |
|-------|----------|------------|
| Baseline (majority class) | ~47% | No learning — always picks most common class |
| **Logistic Regression** | **~83%** | ✅ Best performer — fast, interpretable, robust on TF-IDF |
| Random Forest | ~81% | Good but slower; trees don't add much over LR on sparse data |
| Decision Tree | ~76% | Worst ML model — overfits on high-dimensional TF-IDF features |

---

### 🧠 What We Learned

1. **Text Preprocessing Is Critical** — Raw tweets are noisy. Cleaning, normalization, stopword removal, and stemming all contribute to model performance.

2. **TF-IDF Is Powerful and Simple** — It captures word importance without any deep learning, and works especially well with linear models like Logistic Regression.

3. **Linear > Tree on Sparse Text** — For high-dimensional sparse features (TF-IDF), linear models often outperform tree-based ones. Trees struggle to find useful splits in 50,000-dimensional space.

4. **Ensemble Methods Help — But Not Always** — Random Forest improves over a single Decision Tree, but still trails Logistic Regression on text.

5. **Always Beat Your Baseline** — A model that achieves 83% when the baseline is 47% is genuinely useful. A model that achieves 83% when the baseline is 80% is barely helping.

---

### 🚀 Next Steps & Improvements

| Technique | Expected Improvement |
|-----------|---------------------|
| Use **Transformer models** (BERT, RoBERTa) | Large accuracy gains — state of the art |
| Use **word embeddings** (Word2Vec, GloVe) | Capture semantic similarity |
| **Hyperparameter tuning** (GridSearch) | Squeeze out 2–5% extra accuracy |
| **Data augmentation** | Improve robustness on minority classes |
| Use **character-level features** | Handle slang, abbreviations, misspellings |

---

### 📚 References
- Sentiment140 Dataset — [Kaggle](https://www.kaggle.com/datasets/abhi8923shriv/sentiment-analysis-dataset)
- Hugging Face Sentiment Analysis Guide — [HuggingFace Blog](https://huggingface.co/blog/sentiment-analysis-python)
- NLTK Documentation — [nltk.org](https://www.nltk.org/)
- Scikit-learn TF-IDF — [scikit-learn.org](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- https://www.kaggle.com/code/mohsinsial/sentiment-analysis-by-nlp